In [1]:
import sys
sys.path.insert(0, '..')

import polars as pl
from diaphanous.show import show

pl.Config.set_thousands_separator(",")

%load_ext rpy2.ipython
import diaphanous.arr as arr
arr.install()
RLIB = arr.RLIB

import rpy2.robjects as ro
from rpy2.robjects.packages import importr
from rpy2.robjects import pandas2ri
import rpy2.rinterface as ri

ro.globalenv['RLIB'] = RLIB
ro.r(".libPaths(RLIB)")

In [2]:
reports = pl.read_csv("../data/ocse-reports-per-year.csv")
population = pl.read_csv("../data/populations-simple.csv")
internet_users = pl.read_csv("../data/internet-users-un.csv")
social_media_accounts = pl.read_csv("../data/social-accounts.csv")

frame = reports.join(
    population, on="year", how="left"
).join(
    internet_users, on="year", how="left"
).join(
    social_media_accounts, on="year", how="left"
).select(
    pl.col("year", "reports", "population"),
    (pl.col("million_accounts") * 1_000_000).alias("accounts"),
    (pl.col("internet_users_pct") * pl.col("population") / 100).alias("internet_users"),
).filter(
    (pl.col("year") <= 2024) & (pl.col("year") >= 2014)
).drop_nans()

corrs = frame.corr().select(
    pl.col("internet_users", "population", "accounts")
).row(1)
print(corrs)

reports_to_2023 = reports.filter(pl.col("year") <= 2023).to_pandas()
decade_plus = frame.to_pandas()

show(reports_to_2023)
show(decade_plus)

(0.9043716685081888, 0.8975662870668343, 0.9064924832012359)


,year,reports
0,"2,023","36,210,368"
1,"2,022","32,059,029"
2,"2,021","29,397,681"
3,"2,020","21,751,085"
4,"2,019","16,987,361"
5,"2,018","18,462,422"
6,"2,017","10,214,753"
7,"2,016","8,297,923"
8,"2,015","4,403,657"
9,"2,014","1,106,071"


,year,reports,population,accounts,internet_users
0,"2,024","20,512,803","8,161,972,572.5","5,037,000,000.0","5,517,493,459.0"
1,"2,023","36,210,368","8,091,734,930.0","4,770,000,000.0","5,291,994,644.2"
2,"2,022","32,059,029","8,021,407,192.0","4,632,000,000.0","5,109,636,381.3"
3,"2,021","29,397,681","7,954,448,391.5","4,214,000,000.0","4,907,894,657.6"
4,"2,020","21,751,085","7,887,001,292.0","3,726,000,000.0","4,621,782,757.1"
5,"2,019","16,987,361","7,811,293,698.5","3,478,000,000.0","4,132,174,366.5"
6,"2,018","18,462,422","7,729,902,780.5","3,212,000,000.0","3,749,002,848.5"
7,"2,017","10,214,753","7,645,617,954.0","2,804,000,000.0","3,455,819,315.2"
8,"2,016","8,297,923","7,558,554,525.5","2,320,000,000.0","3,235,061,336.9"
9,"2,015","4,403,657","7,470,491,871.5","2,094,000,000.0","2,973,255,764.9"


In [3]:
stats = importr('stats')
base = importr('base')

with (ro.default_converter + pandas2ri.converter).context():
    r_to_2023 = ro.conversion.get_conversion().py2rpy(reports_to_2023)
    r_decade_plus = ro.conversion.get_conversion().py2rpy(decade_plus)

ro.globalenv['reports_to_2023'] = r_to_2023
ro.globalenv['decade_plus'] = r_decade_plus

In [4]:
ro.r("library(dplyr)")
ro.r("library(segmented)")

show("<h2>All Years <= 2023: Single Exponential</h2>")
ro.r("mod_exp <- lm(log(reports) ~ year, data = reports_to_2023)")
print(ro.r("mod_exp"))
print(ro.r("summary(mod_exp)"))

show("<h2>All Years <= 2023: Piecewise Linear</h2>")
ro.r("mod_lin <- lm(reports ~ year, data = reports_to_2023)")
ro.r("mod_seg <- segmented(mod_lin, psi=c(2014))") #, control = seg.control(it.max = 0))
print(ro.r("summary(mod_seg)"))

show("<h2>All Years <= 2023: Piecewise Linear, Fixed Break at 2014</h2>")
ro.r("mod_lin <- lm(reports ~ year, data = reports_to_2023)")
ro.r("mod_seg <- segmented(mod_lin, psi=c(2014), control = seg.control(it.max = 0))")
print(ro.r("summary(mod_seg)"))

show("<h2>2014–2023: Linear Model</h2>")
ro.r("decade <- decade_plus |> filter(year <= 2023)")
ro.r("mod_lin2 <- lm(reports ~ year, data = decade)")
print(ro.r("summary(mod_lin2)"))



R[write to console]: 
Attaching package: ‘dplyr’


R[write to console]: The following objects are masked from ‘package:stats’:

    filter, lag


R[write to console]: The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


R[write to console]: Loading required package: MASS

R[write to console]: 
Attaching package: ‘MASS’


R[write to console]: The following object is masked from ‘package:dplyr’:

    select


R[write to console]: Loading required package: nlme

R[write to console]: 
Attaching package: ‘nlme’


R[write to console]: The following object is masked from ‘package:dplyr’:

    collapse





Call:
lm(formula = log(reports) ~ year, data = reports_to_2023)

Coefficients:
(Intercept)         year  
    -710.54         0.36  



Call:
lm(formula = log(reports) ~ year, data = reports_to_2023)

Residuals:
     Min       1Q   Median       3Q      Max 
-0.95745 -0.49054  0.04278  0.48085  0.84256 

Coefficients:
              Estimate Std. Error t value Pr(>|t|)    
(Intercept) -710.54007   31.73968  -22.39   <2e-16 ***
year           0.35997    0.01579   22.80   <2e-16 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 0.6037 on 24 degrees of freedom
Multiple R-squared:  0.9559,	Adjusted R-squared:  0.954 
F-statistic: 519.9 on 1 and 24 DF,  p-value: < 2.2e-16





	***Regression Model with Segmented Relationship(s)***

Call: 
segmented.lm(obj = mod_lin, psi = c(2014))

Estimated Break-Point(s):
             Est. St.Err
psi1.year 2014.2  0.231

Coefficients of the linear terms:
             Estimate Std. Error t value Pr(>|t|)
(Intercept) -83472672  107129154  -0.779    0.444
year            41710      53404   0.781    0.443
U1.year       3961035     149150  26.557       NA

Residual standard error: 1079000 on 22 degrees of freedom
Multiple R-Squared: 0.9921,  Adjusted R-squared: 0.991 

Boot restarting based on 6 samples. Last fit:
Convergence attained in 2 iterations (rel. change 1.1175e-10)




Call:
lm(formula = reports ~ year + U1.year, data = mfExt)

Residuals:
     Min       1Q   Median       3Q      Max 
-2884575   -48072    14883   183349  2504636 

Coefficients:
             Estimate Std. Error t value Pr(>|t|)    
(Intercept) -41718310   94620204  -0.441    0.663    
year            20864      47150   0.443    0.662    
U1.year       3893286     125291  31.074   <2e-16 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 1072000 on 23 degrees of freedom
Multiple R-squared:  0.9918,	Adjusted R-squared:  0.9911 
F-statistic:  1397 on 2 and 23 DF,  p-value: < 2.2e-16





Call:
lm(formula = reports ~ year, data = decade)

Residuals:
     Min       1Q   Median       3Q      Max 
-2865792 -1279076   342308   831642  2537505 

Coefficients:
              Estimate Std. Error t value Pr(>|t|)    
(Intercept) -7.911e+09  4.030e+08  -19.63 4.72e-08 ***
year         3.928e+06  1.997e+05   19.67 4.64e-08 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 1814000 on 8 degrees of freedom
Multiple R-squared:  0.9797,	Adjusted R-squared:  0.9772 
F-statistic: 387.1 on 1 and 8 DF,  p-value: 4.636e-08




In [5]:
ro.r("library(estimatr)")

r2_adj = {}
robust_r2_adj = {}

MODELS = [
    ('year', 'reports ~ year'),
    ('pop', 'reports ~ population'),
    ('inet', 'reports ~ internet_users'),
    ('social', 'reports ~ accounts'),
    ('social_year', 'reports ~ accounts + year'),
]

for name, relation in MODELS:
    ro.r(f'mod_{name} <- lm({relation}, data = decade)')
    r2_adj[name] = ro.r(f'summary(mod_{name})$adj.r.squared')
    ro.r(f'robust_mod_{name} <- lm_robust({relation}, data = decade)')
    robust_r2_adj[name] = ro.r(f'summary(robust_mod_{name})$adj.r.squared')

show("<h2>Normality</h2>")

for name, _ in MODELS:
    print(ro.r(f'shapiro.test(rstandard(mod_{name}))'))

show("<h2>Adjusted R²</h2>")
for key, value in sorted(r2_adj.items(), key=lambda i: i[1][0]):
    print(f"{key}: {value}")

show("<h3>Best Linear Model: Social Media Accounts</h3>")
print(ro.r('mod_social'))
print(ro.r('summary(mod_social)'))

show("<h2>Robust Adjusted R²</h2>")
for key, value in sorted(robust_r2_adj.items(), key=lambda i: i[1][0]):
    print(f"{key}: {value}")

show("<h3>Best Linear Model: Social Media Accounts</h3>")
print(ro.r('robust_mod_social'))
print(ro.r('summary(robust_mod_social)'))


	Shapiro-Wilk normality test

data:  rstandard(mod_year)
W = 0.93124, p-value = 0.4602



	Shapiro-Wilk normality test

data:  rstandard(mod_pop)
W = 0.82879, p-value = 0.03236



	Shapiro-Wilk normality test

data:  rstandard(mod_inet)
W = 0.95526, p-value = 0.7307



	Shapiro-Wilk normality test

data:  rstandard(mod_social)
W = 0.92137, p-value = 0.3685



	Shapiro-Wilk normality test

data:  rstandard(mod_social_year)
W = 0.92703, p-value = 0.4193




inet: [1] 0.9639632

pop: [1] 0.9664247

year: [1] 0.9772185

social_year: [1] 0.9778598

social: [1] 0.9797739




Call:
lm(formula = reports ~ accounts, data = decade)

Coefficients:
(Intercept)     accounts  
 -2.010e+07    1.147e-02  



Call:
lm(formula = reports ~ accounts, data = decade)

Residuals:
     Min       1Q   Median       3Q      Max 
-2806881  -950845   125834  1487770  1786217 

Coefficients:
              Estimate Std. Error t value Pr(>|t|)    
(Intercept) -2.010e+07  1.896e+06   -10.6 5.48e-06 ***
accounts     1.147e-02  5.487e-04    20.9 2.88e-08 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 1709000 on 8 degrees of freedom
Multiple R-squared:  0.982,	Adjusted R-squared:  0.9798 
F-statistic:   437 on 1 and 8 DF,  p-value: 2.878e-08




inet: [1] 0.9639632

pop: [1] 0.9664247

year: [1] 0.9772185

social_year: [1] 0.9778598

social: [1] 0.9797739



                 Estimate   Std. Error   t value     Pr(>|t|)      CI Lower
(Intercept) -2.009925e+07 1.440713e+06 -13.95090 6.751501e-07 -2.342154e+07
accounts     1.147024e-02 4.324234e-04  26.52548 4.387673e-09  1.047307e-02
                 CI Upper DF
(Intercept) -1.677696e+07  8
accounts     1.246741e-02  8


Call:
lm_robust(formula = reports ~ accounts, data = decade)

Standard error type:  HC2 

Coefficients:
              Estimate Std. Error t value  Pr(>|t|)   CI Lower   CI Upper DF
(Intercept) -2.010e+07  1.441e+06  -13.95 6.752e-07 -2.342e+07 -1.678e+07  8
accounts     1.147e-02  4.324e-04   26.53 4.388e-09  1.047e-02  1.247e-02  8

Multiple R-squared:  0.982 ,	Adjusted R-squared:  0.9798 
F-statistic: 703.6 on 1 and 8 DF,  p-value: 4.388e-09

